#VGG

In [ ]:
import torch
import torch.nn as nn

class VGGBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_convs):
        """
        A single block in the VGG network.
        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels(Kernels).
            num_convs: Number of convolutional layers in the block.
        """
        super(VGGBlock, self).__init__()
        layers = []
        for _ in range(num_convs):
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            layers.append(nn.ReLU(inplace=True))
            in_channels = out_channels
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))  # Add max pooling
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


In [ ]:
class VGG(nn.Module):
    def __init__(self, architecture, num_classes=4, input_size=(3, 224, 224)):
        """
        VGG model using VGG blocks, with dynamic classifier size detection.
        Args:
            architecture: List of tuples defining the VGG architecture.
                          Each tuple is (num_convs, out_channels).
            num_classes: Number of output classes for classification.
            input_size: Tuple representing the input size (C, H, W).
        """
        super(VGG, self).__init__()
        self.features = self.create_vgg_layers(architecture)

        # Dynamically calculate the flattened size
        self.flattened_size = self._get_flattened_size(input_size)

        # Define the classifier with the dynamically detected size
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flattened_size, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def _get_flattened_size(self, input_size):
        """
        Calculate the size of the flattened tensor after passing through the feature extractor.
        This allows creating varying input size models and enables constructing variable sized feature extractors.
        Args:
            input_size: Tuple representing the input size (C, H, W).
        Returns:
            int: Flattened size of the tensor.
        """
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_size)  # Batch size 1
            output = self.features(dummy_input)
            return output.numel()  # input size for FC layer

    @staticmethod
    def create_vgg_layers(architecture):
        layers = []
        in_channels = 3  # Starting with RGB images (3 channels)
        for num_convs, out_channels in architecture:
            layers.append(VGGBlock(in_channels, out_channels, num_convs))
            in_channels = out_channels
        return nn.Sequential(*layers)


In [ ]:
def vgg16_architecture():
    """
    Returns the architecture for VGG-16.
    Format: [(num_convs, out_channels), ...]
    """
    return [
        (2, 64),  # Block 1: 2 conv layers, 64 filters
        (2, 128),  # Block 2: 2 conv layers, 128 filters
        (3, 256),  # Block 3: 3 conv layers, 256 filters
        (3, 512),  # Block 4: 3 conv layers, 512 filters
        (3, 512),  # Block 5: 3 conv layers, 512 filters
    ]
